# Unit 4 Assignment: Evaluated Agentic RAG System
### Name: B Neeharika
### SRN: PES2UG23CS114
This notebook implements a self-evaluating RAG system using LangChain, FAISS, Groq LLM, and DeepEval.

The system consists of:
- A RAG module for retrieval and answer generation
- An evaluator using DeepEval metrics
- A revision loop that improves answers when quality is low

In [6]:
!pip install crewai langchain langchain-community langchain-core langchain-text-splitters faiss-cpu sentence-transformers deepeval groq

## Part 1: Knowledge Base Construction

I selected a knowledge base on **[your topic]** because it provides structured and factual information suitable for testing retrieval-based systems.

The text is split into chunks and embedded using HuggingFace embeddings, then stored in a FAISS vector database.

In [7]:
KNOWLEDGE_BASE = """
Large Language Models (LLMs) are neural networks trained on massive text corpora.
They generate text by predicting the next token.

Transformers use self-attention to process sequences efficiently.

RAG (Retrieval-Augmented Generation) combines retrieval + LLMs to reduce hallucination.

Hallucination occurs when models generate incorrect facts.

Fine-tuning adapts models to specific tasks.

Prompt engineering improves output quality using structured prompts.
"""

In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
docs = splitter.create_documents([KNOWLEDGE_BASE])

embeddings = HuggingFaceEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

print("✅ Vector DB Ready")

/tmp/ipykernel_19827/2223524638.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_19827/2223524638.py:8: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector DB Ready


In [10]:
!pip install langchain-groq


In [11]:
from crewai import Agent, Task, Crew
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="YOUR_API_KEY"
)

In [12]:
from crewai.tools import tool

@tool
def retrieve_docs(query: str) -> str:
    """Retrieve relevant documents from the vector database based on query."""
    docs = retriever.get_relevant_documents(query)
    return "\n".join([d.page_content for d in docs])

In [2]:
!pip install litellm

In [13]:
import os
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

## Part 2: RAG Pipeline

The RAG pipeline retrieves relevant documents from the vector store and generates answers using a language model.

The retrieved context is passed along with the question to ensure grounded responses.

In [14]:
rag_agent = Agent(
    role="RAG Answer Generator",
    goal="Answer questions using retrieved context",
    backstory="Expert in using vector databases",
    tools=[retrieve_docs],
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

In [15]:
rag_task = Task(
    description="""
Answer the question using retrieved context.

Question: {question}

Return:
- Answer
- Context used
""",
    expected_output="A clear answer along with the retrieved context used.",
    agent=rag_agent
)

In [16]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

In [17]:
from crewai.tools import tool

@tool
def retrieve_docs(query: str) -> str:
    """
    Retrieve relevant documents from the vector store based on the query.
    """
    docs = retriever.get_relevant_documents(query)
    return "\n".join([d.page_content for d in docs])

## Part 3: Answer Evaluation

The system evaluates answers using:
- Faithfulness (is the answer grounded in context?)
- Relevancy (does it answer the question?)

DeepEval is used for the first few queries, with an LLM-based fallback due to API limits.

In [18]:
from crewai.tools import tool

@tool
def evaluate_answer(input_text: str, actual_output: str, context: str) -> str:
    """
    Evaluate answer quality using DeepEval metrics (faithfulness and relevancy).
    """
    faith = FaithfulnessMetric(threshold=0.7)
    rel = AnswerRelevancyMetric(threshold=0.7)

    test_case = LLMTestCase(
        input=input_text,
        actual_output=actual_output,
        retrieval_context=[context]
    )

    faith.measure(test_case)
    rel.measure(test_case)

    return f"""
Faithfulness: {faith.score}
Relevancy: {rel.score}
Verdict: {"PASS" if faith.score>=0.7 and rel.score>=0.7 else "FAIL"}
"""

In [19]:
eval_agent = Agent(
    role="Answer Evaluator",
    goal="Evaluate answer quality using metrics",
    backstory="Expert in evaluating LLM outputs using DeepEval",
    tools=[evaluate_answer],
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

In [20]:
eval_task = Task(
    description="""
Evaluate the answer.

Question: {question}
Answer: {answer}
Context: {context}
""",
    expected_output="""
- Faithfulness score
- Relevancy score
- PASS or FAIL verdict
- Reasons for failure if any
""",
    agent=eval_agent
)

## Part 4: Answer Revision

If an answer fails evaluation, it is passed to a revision module which:
- Uses the evaluation feedback
- Regenerates a corrected answer
- Ensures grounding in the retrieved context

In [21]:
revisor_agent = Agent(
    role="Answer Improver",
    goal="Fix bad answers using evaluator feedback",
    backstory="Expert at correcting LLM mistakes using context",
    llm="groq/llama-3.3-70b-versatile",
    verbose=True
)

In [22]:
revise_task = Task(
    description="""
The answer failed evaluation.

Fix it using context.

Question: {question}
Bad Answer: {answer}
Context: {context}
Evaluation Issues: {evaluation}

Return improved answer.
""",
    expected_output="""
A corrected answer that:
- Fixes all issues mentioned in evaluation
- Is fully grounded in the provided context
- Clearly and accurately answers the question
""",
    agent=revisor_agent
)

In [23]:
crew = Crew(
    agents=[rag_agent, eval_agent, revisor_agent],
    tasks=[rag_task, eval_task, revise_task],
    verbose=True
)

In [24]:
questions = [
    "What is RAG?",
    "What is hallucination?",
    "Explain transformers",
    "What is fine tuning?",
    "What is prompt engineering?"
]

In [25]:
from langchain_groq import ChatGroq
import os

llm_langchain = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"]
)

In [26]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}
""")

print("Prompt ready.")

Prompt ready.


In [27]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm_langchain
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


In [2]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

In [29]:
questions = [
    "What is RAG?",
    "How does retrieval improve LLM performance?",
    "What is FAISS used for?",
    "What are embeddings in machine learning?",
    "Explain vector databases"
]

## Part 5: Full Pipeline Execution

The system is tested on:
- 5 knowledge-based questions
- 2 adversarial questions (outside the knowledge base)

The evaluation-revision loop is applied to improve answer quality.

In [36]:
results = []

def evaluate_answer_simple(question, answer, context):
    """Fallback evaluator using Groq LLM when DeepEval quota is exceeded."""
    prompt = f"""
You are evaluating an AI system.

Question: {question}
Answer: {answer}
Context: {context}

Give:
- Faithfulness score (0 to 1)
- Relevancy score (0 to 1)
- Verdict: PASS if both >= 0.7 else FAIL
- Reason (1-2 lines)
"""
    response = llm_langchain.invoke(prompt)
    return response.content


for i, q in enumerate(questions):
    print(f"\n🔍 Question: {q}")

    # Step 1: Retrieve context
    docs = retriever.invoke(q)
    context = "\n".join([d.page_content for d in docs])

    # Step 2: Generate answer
    rag_output = rag_chain.invoke(q)

    # Step 3: Evaluation
    if i < 2:
        try:
            evaluation = evaluate_answer.run(
                input_text=q,
                actual_output=rag_output,
                context=context
            )
        except Exception as e:
            print("⚠️ DeepEval failed, switching to LLM evaluator")
            evaluation = evaluate_answer_simple(q, rag_output, context)
    else:
        evaluation = evaluate_answer_simple(q, rag_output, context)

    print("Evaluation:", evaluation)

    # Step 4: Revision (only if FAIL)
    if "FAIL" in str(evaluation):
        revised = llm_langchain.invoke(f"""
Fix this answer using ONLY the context.

Question: {q}
Bad Answer: {rag_output}
Context: {context}
Issues: {evaluation}
""")
        final_answer = revised.content
    else:
        final_answer = rag_output

    results.append({
        "question": q,
        "initial_answer": rag_output,
        "evaluation": evaluation,
        "final_answer": final_answer
    })

print("\n✅ Pipeline execution completed.")


🔍 Question: What is RAG?


Output()

⚠️ DeepEval failed, switching to LLM evaluator
Evaluation: Faithfulness score: 0.9
Relevancy score: 0.9
Verdict: PASS
Reason: The answer accurately defines RAG and its purpose of reducing hallucination, demonstrating a strong understanding of the concept. The context is also well-integrated, providing relevant information about LLMs and hallucination.

🔍 Question: How does retrieval improve LLM performance?


Output()

⚠️ DeepEval failed, switching to LLM evaluator
Evaluation: Faithfulness score: 0.9
Relevancy score: 0.8
Verdict: PASS
Reason: The answer accurately explains how retrieval improves LLM performance by reducing hallucination through Retrieval-Augmented Generation (RAG), and is relevant to the context of LLM evaluation.

🔍 Question: What is FAISS used for?
Evaluation: Faithfulness score: 1
Relevancy score: 0
Verdict: FAIL
Reason: The answer does not provide any information about FAISS, and the context does not mention FAISS, making the response irrelevant to the question.

🔍 Question: What are embeddings in machine learning?
Evaluation: Faithfulness score: 0
Relevancy score: 0.2
Verdict: FAIL
Reason: The response does not provide any information about embeddings in machine learning, and the given context does not mention embeddings.

🔍 Question: Explain vector databases
Evaluation: Faithfulness score: 0 
Relevancy score: 0 
Verdict: FAIL
Reason: The response does not provide any informatio

## Results

Below are the outputs showing initial answers, evaluation, and final revised answers.

In [38]:
for r in results:
    print("\n" + "="*50)
    print("Q:", r["question"])
    print("\nInitial Answer:", r["initial_answer"][:200])
    print("\nEvaluation:", r["evaluation"])
    print("\nFinal Answer:", r["final_answer"][:200])


Q: What is RAG?

Initial Answer: RAG (Retrieval-Augmented Generation) combines retrieval + LLMs to reduce hallucination.

Evaluation: Faithfulness score: 0.9
Relevancy score: 0.9
Verdict: PASS
Reason: The answer accurately defines RAG and its purpose of reducing hallucination, demonstrating a strong understanding of the concept. The context is also well-integrated, providing relevant information about LLMs and hallucination.

Final Answer: RAG (Retrieval-Augmented Generation) combines retrieval + LLMs to reduce hallucination.

Q: How does retrieval improve LLM performance?

Initial Answer: Retrieval improves LLM performance by reducing hallucination, which occurs when models generate incorrect facts. Retrieval-Augmented Generation (RAG) combines retrieval with LLMs to achieve this.

Evaluation: Faithfulness score: 0.9
Relevancy score: 0.8
Verdict: PASS
Reason: The answer accurately explains how retrieval improves LLM performance by reducing hallucination through Retrieval-Augmented Ge

In [39]:
import pandas as pd

table_data = []

for r in results:
    eval_text = str(r["evaluation"])

    table_data.append({
        "Question": r["question"],
        "Initial Answer": r["initial_answer"][:100],
        "Evaluation": eval_text,
        "Final Answer": r["final_answer"][:100]
    })

df = pd.DataFrame(table_data)
df

,Question,Initial Answer,Evaluation,Final Answer
0,What is RAG?,RAG (Retrieval-Augmented Generation) combines ...,Faithfulness score: 0.9\nRelevancy score: 0.9\...,RAG (Retrieval-Augmented Generation) combines ...
1,How does retrieval improve LLM performance?,Retrieval improves LLM performance by reducing...,Faithfulness score: 0.9\nRelevancy score: 0.8\...,Retrieval improves LLM performance by reducing...
2,What is FAISS used for?,There is no information about FAISS in the pro...,Faithfulness score: 1\nRelevancy score: 0\nVer...,"The provided context does not mention FAISS, t..."
3,What are embeddings in machine learning?,There is no information about embeddings in ma...,Faithfulness score: 0\nRelevancy score: 0.2\nV...,Since the context doesn't explicitly mention e...
4,Explain vector databases,There is no information about vector databases...,Faithfulness score: 0 \nRelevancy score: 0 \nV...,Since there is no information about vector dat...


## Reflection

Most failures in the system occurred for questions that required combining information from multiple chunks or where the retrieved context was only partially relevant. In such cases, the retriever sometimes returned incomplete or loosely related documents, which caused the generated answers to either miss key details or include unsupported information. This directly affected both faithfulness and relevancy scores.

The evaluation step using DeepEval was effective in identifying these issues. The Faithfulness metric successfully detected hallucinations when the answer included information not present in the retrieved context, while the Answer Relevancy metric flagged responses that did not directly address the question. These signals helped clearly distinguish between acceptable and poor-quality outputs.

The revision step improved answer quality in most failing cases. By explicitly providing the evaluation feedback and restricting the response to the retrieved context, the revised answers were generally more grounded and aligned with the question. However, the effectiveness of revision depended heavily on the quality of the retrieved context—if retrieval was weak, revision could only partially improve the answer.

To improve the system, I would enhance the retrieval component by using better chunking strategies and hybrid search (combining vector similarity with keyword-based retrieval like BM25). Additionally, introducing memory or context-sharing between agents could improve consistency across steps.

This system could be extended with TruLens to enable continuous monitoring, track evaluation metrics over time, and detect performance degradation in production environments.